## 1. Постановка задачі

**1. Прикладна задача регресії**
Розв'язується задача прогнозування неперервної кількісної характеристики біологічного об'єкта (квітки ірису) на основі інших його просторових вимірів та біологічного виду.

**2. Визначення складових задачі:**
* **Об'єкт спостереження:** Окремий екземпляр квітки роду Ірис (*Iris*).
* **Вхідні ознаки:** Довжина чашолистка (`sepal_length`), довжина пелюстки (`petal_length`), ширина пелюстки (`petal_width`) — числові; вид квітки (`species`) — категоріальна ознака.
* **Цільова змінна:** `sepal_width` (ширина чашолистка квітки), вимірюється у сантиметрах.
* **Практичний зміст прогнозу:** Реконструкція втрачених геометричних параметрів пошкоджених гербарних зразків та біометричний контроль у селекції (виявлення аномальних пропорцій розвитку квітки).

**3. Основна експериментальна гіпотеза**
Чи дозволяють регресійні моделі машинного навчання (лінійні та нелінійні) отримати суттєво меншу похибку прогнозування ширини чашолистка порівняно з наївною baseline-моделлю (прогнозуванням середнього значення), і чи виправдане ускладнення моделі для цього набору даних?

**4. Основна метрика для порівняння**
Як основну метрику обрано **RMSE** (Root Mean Squared Error). 
* **Обґрунтування:** RMSE обчислюється і виводиться у тих самих фізичних одиницях вимірювання, що й цільова змінна (у сантиметрах). Це робить похибку легко інтерпретованою для спеціалістів предметної області (ботаніків). Крім того, математичні властивості метрики (через піднесення до квадрата перед добуванням кореня) сильніше штрафують модель за великі відхилення, що є критично важливим для уникнення грубих помилок при біометричних вимірюваннях.

## 2. Формування навчальної та тестової вибірок

Для забезпечення об'єктивного оцінювання якості моделей, набір даних розділяється на навчальну (Train) та тестову (Test) підмножини.

In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Завантаження набору даних (якщо не завантажено раніше)
iris = load_iris(as_frame=True)
df = iris.frame
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
df['species'] = df['species'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

# Виокремлення вхідних ознак та цільової змінної
X = df.drop(columns=['sepal_width'])
y = df['sepal_width']

# Розбиття даних на навчальну та тестову вибірки (80% / 20%)
# random_state фіксується для відтворюваності результатів експерименту
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=X['species'] # Збереження балансу видів у вибірках
)

# Виведення розмірів отриманих вибірок
print(f"Розмір початкового датасету: X={X.shape}, y={y.shape}")
print(f"Розмір навчальної вибірки (Train): X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Розмір тестової вибірки (Test): X_test={X_test.shape}, y_test={y_test.shape}")

Розмір початкового датасету: X=(150, 4), y=(150,)
Розмір навчальної вибірки (Train): X_train=(120, 4), y_train=(120,)
Розмір тестової вибірки (Test): X_test=(30, 4), y_test=(30,)


**Важливе зауваження щодо тестової вибірки:**
Отримана вибірка `X_test` та `y_test` є повністю ізольованою. Вона **категорично не використовуватиметься** на етапах попередньої обробки даних (наприклад, для розрахунку середніх значень при масштабуванні), вибору архітектури моделей, налаштування гіперпараметрів чи крос-валідації. Її єдине призначення — фінальне незалежне тестування найкращої моделі наприкінці експерименту.